# D1: Surface Classifier + Routing Experiment (v2.0)

## Цель
Проверить гипотезу: поверхностный классификатор достигает F1 >= 0.85 и снижает LLM-вызовы.

## Обновления v2.0
- 8 классов intent (вместо 6)
- 600+ samples (вместо 120)
- Sentence-Transformers (rubert-tiny2)
- SetFit few-shot learning
- GridSearchCV для оптимизации
- Каскадный классификатор (Rule → ML → LLM)

## Классы intent (8)
| Intent | Описание | Scenario |
|--------|----------|----------|
| `booking` | Запись на приём | booking_flow |
| `complaint_primary` | Первичная жалоба | anamnesis_flow |
| `price_question` | Вопросы о стоимости | faq_flow |
| `reschedule_cancel` | Перенос/отмена | reschedule_flow |
| `followup_question` | Уточнения по лечению | faq_flow |
| `clinic_faq` | Вопросы о клинике | faq_flow |
| `visit_recommendations` | Рекомендации до/после | faq_flow |
| `other` | Приветствия, благодарности | faq_flow |

## 1. Setup

In [3]:
!pip install -r requirements.txt

  Using cached sentence_transformers-5.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
Using cached sentence_transformers-5.2.0-py3-none-any.whl (493 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 6.1 MB/s  0:00:00m eta 0:00:01
Using cached evaluate-0.4.6-py3-none-any.whl (84 kB)
Using cached colorlog-6.10.1-py3-none-any.whl (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [setfit] 8/10 [sentence-transformers]


In [4]:
# Imports
import sys
import os
import random
import json
import re
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm

# ML
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, accuracy_score
import joblib

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Load environment
from dotenv import load_dotenv
load_dotenv()

# Suppress warnings
warnings.filterwarnings('ignore')

# Add utils to path
sys.path.insert(0, str(Path('.').resolve()))
from utils import (
    generate_d1_dataset, load_or_generate_d1,
    compute_classification_metrics, compute_economics,
    plot_confusion_matrix, plot_f1_by_class, plot_economics_comparison,
    TogetherLLM, INTENT_LABELS, INTENT_TO_SCENARIO,
    augment_dataset, generate_d1_with_augmentation,
    EmbeddingClassifier, SetFitClassifier, RuleBasedClassifier, CascadeClassifier,
    create_tfidf_pipeline, get_tfidf_param_grid, run_gridsearch
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Paths
DATA_PATH = Path('data/d1_messages_v2.csv')
OUTPUT_FIGURES = Path('outputs/figures')
OUTPUT_TABLES = Path('outputs/tables')
OUTPUT_REPORTS = Path('outputs/reports')
MODELS_PATH = Path('models')

# Ensure directories exist
for p in [OUTPUT_FIGURES, OUTPUT_TABLES, OUTPUT_REPORTS, MODELS_PATH, DATA_PATH.parent]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Experiment started: {datetime.now().isoformat()}")
print(f"Random seed: {SEED}")
print(f"Intent classes: {len(INTENT_LABELS)} - {INTENT_LABELS}")

Experiment started: 2026-01-24T00:48:58.346742
Random seed: 42
Intent classes: 8 - ['booking', 'complaint_primary', 'price_question', 'reschedule_cancel', 'followup_question', 'clinic_faq', 'visit_recommendations', 'other']


## 2. Intent Classes Definition

In [5]:
# Load label map from config
with open('configs/label_map.json', 'r', encoding='utf-8') as f:
    label_config = json.load(f)

# Display intent classes
print("\n=== 8 Intent Classes ===")
for intent, info in label_config['intents'].items():
    print(f"  {info['id']}. {intent}: {info['name_ru']} - {info['description']}")

print("\n=== Intent → Scenario Mapping ===")
for intent, scenario in label_config['intent_to_scenario'].items():
    print(f"  {intent} → {scenario}")

# Scenario labels
SCENARIO_LABELS = list(set(INTENT_TO_SCENARIO.values()))
print(f"\nScenarios: {SCENARIO_LABELS}")


=== 8 Intent Classes ===
  0. booking: Запись на приём - Пациент хочет записаться на приём
  1. complaint_primary: Первичная жалоба - Пациент описывает симптомы или проблему
  2. followup_question: Уточняющий вопрос - Пациент задаёт дополнительный вопрос о процедуре или лечении
  3. price_question: Вопрос о цене - Пациент интересуется стоимостью услуг
  4. reschedule_cancel: Перенос/отмена - Пациент хочет перенести или отменить запись
  5. clinic_faq: Вопросы о клинике - Вопросы о местоположении, часах работы, оборудовании, персонале
  6. visit_recommendations: Рекомендации до/после - Вопросы о подготовке к процедуре и уходе после
  7. other: Прочее - Приветствия, благодарности, подтверждения и прочее

=== Intent → Scenario Mapping ===
  booking → booking_flow
  complaint_primary → anamnesis_flow
  followup_question → faq_flow
  price_question → faq_flow
  reschedule_cancel → reschedule_flow
  clinic_faq → faq_flow
  visit_recommendations → faq_flow
  other → faq_flow

Scenarios: ['bo

## 3. Data Loading (600+ samples)

In [6]:
# Generate fresh dataset with 600 samples (8 classes)
print("Generating dataset with 600 samples...")
df = generate_d1_dataset(n=600, seed=SEED)

# Save to file
df.to_csv(DATA_PATH, index=False)
print(f"Saved to: {DATA_PATH}")

print(f"\nDataset size: {len(df)}")
print(f"Source: {df['source'].value_counts().to_dict()}")
print(f"\nIntent distribution:")
print(df['label_intent'].value_counts())

Generating dataset with 600 samples...
Saved to: data/d1_messages_v2.csv

Dataset size: 588
Source: {'synthetic': 588}

Intent distribution:
label_intent
visit_recommendations    75
price_question           75
complaint_primary        75
booking                  75
followup_question        75
clinic_faq               75
reschedule_cancel        75
other                    63
Name: count, dtype: int64


In [7]:
# Show sample data by class
print("\n=== Sample Messages by Class ===")
for intent in INTENT_LABELS:
    samples = df[df['label_intent'] == intent]['text'].head(3).tolist()
    print(f"\n[{intent}]:")
    for s in samples:
        print(f"  - {s}")


=== Sample Messages by Class ===

[booking]:
  - хотела бы записаться к детскому стоматологу
  - Хочу к ортодонту на среду
  - Нужна запись на лечение каналов

[complaint_primary]:
  - Появились язвочки во рту
  - Болит при надавливании
  - коронка шатается.

[price_question]:
  - Прайс на отбеливание?
  - Почём временная коронка!
  - Почём протезирование?

[reschedule_cancel]:
  - Нужно перенести на пораньше
  - Перенесите запись на четверг
  - Не смогу прийти, занят на работе

[followup_question]:
  - Какие риски у чистку?
  - Какие риски у панорамный снимок?
  - А можно без чистку?

[clinic_faq]:
  - Есть приём в завтра?
  - Принимаете ДМС?
  - Есть ли микроскоп?

[visit_recommendations]:
  - Что нельзя после виниры?
  - Нужно ли приходить повторно
  - Сколько нельзя курить после пластинки?

[other]:
  - привет
  - Приду.
  - доброе утро.


In [8]:
# Train/test split (stratified)
X = df['text'].values
y_intent = df['label_intent'].values
y_scenario = df['label_scenario'].values

X_train, X_test, y_intent_train, y_intent_test, y_scenario_train, y_scenario_test = train_test_split(
    X, y_intent, y_scenario,
    test_size=0.2,
    random_state=SEED,
    stratify=y_intent
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")
print(f"\nTrain class distribution:")
print(pd.Series(y_intent_train).value_counts())

Train size: 470
Test size: 118

Train class distribution:
booking                  60
visit_recommendations    60
complaint_primary        60
followup_question        60
clinic_faq               60
price_question           60
reschedule_cancel        60
other                    50
Name: count, dtype: int64


## 4. EDA (Exploratory Data Analysis)

In [9]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Intent distribution
intent_counts = df['label_intent'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(intent_counts)))
axes[0].bar(intent_counts.index, intent_counts.values, color=colors)
axes[0].set_title('Intent Distribution (8 classes)', fontsize=12)
axes[0].set_xlabel('Intent')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Scenario distribution
scenario_counts = df['label_scenario'].value_counts()
axes[1].pie(scenario_counts.values, labels=scenario_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Scenario Distribution', fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'd1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_FIGURES / 'd1_class_distribution.png'}")

Saved: outputs/figures/d1_class_distribution.png


In [10]:
# Text length analysis
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Text length by class
df.boxplot(column='text_length', by='label_intent', ax=axes[0], rot=45)
axes[0].set_title('Text Length by Intent')
axes[0].set_xlabel('Intent')
axes[0].set_ylabel('Characters')

# Word count by class
df.boxplot(column='word_count', by='label_intent', ax=axes[1], rot=45)
axes[1].set_title('Word Count by Intent')
axes[1].set_xlabel('Intent')
axes[1].set_ylabel('Words')

plt.suptitle('')
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'd1_text_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Baseline A: LLM Classification

In [11]:
# Initialize LLM client
llm = TogetherLLM()

if llm.use_simulator:
    print("WARNING: Using simulator. Check configs/together_config.yaml for API key.")
else:
    print(f"Using real LLM: {llm.model}")
    print(f"API key loaded: {llm.api_key[:20]}...")

Together AI client initialized (via requests). Model: meta-llama/Llama-3.3-70B-Instruct-Turbo
Using real LLM: meta-llama/Llama-3.3-70B-Instruct-Turbo
API key loaded: tgp_v1_ABnz7nPP368yM...


In [12]:
# Run Baseline A on test set (sample for speed)
BASELINE_SAMPLE_SIZE = min(100, len(X_test))  # Limit for API costs
print(f"Running Baseline A on {BASELINE_SAMPLE_SIZE} samples...")

# Sample test set
np.random.seed(SEED)
sample_idx = np.random.choice(len(X_test), BASELINE_SAMPLE_SIZE, replace=False)
X_test_sample = X_test[sample_idx]
y_intent_test_sample = y_intent_test[sample_idx]
y_scenario_test_sample = y_scenario_test[sample_idx]

llm.reset_stats()
baseline_a_predictions = []

for text in tqdm(X_test_sample, desc="Baseline A (LLM)"):
    result = llm.classify_intent(text, INTENT_LABELS)
    baseline_a_predictions.append({
        'text': text,
        'pred_intent': result.get('intent', 'other'),
        'confidence': result.get('confidence', 0.5),
        'tokens_used': result.get('tokens_used', 0),
    })

baseline_a_df = pd.DataFrame(baseline_a_predictions)
baseline_a_df['pred_scenario'] = baseline_a_df['pred_intent'].map(INTENT_TO_SCENARIO)

# Get LLM stats
llm_stats_a = llm.get_stats()
print(f"\nBaseline A LLM Stats:")
print(f"  Total calls: {llm_stats_a['total_calls']}")
print(f"  Total tokens: {llm_stats_a['total_tokens']}")
print(f"  Avg tokens/call: {llm_stats_a['avg_tokens_per_call']:.1f}")

Running Baseline A on 100 samples...


Baseline A (LLM): 100%|██████████| 100/100 [02:29<00:00,  1.49s/it]


Baseline A LLM Stats:
  Total calls: 100
  Total tokens: 21037
  Avg tokens/call: 210.4


In [13]:
# Compute Baseline A metrics
metrics_a_intent = compute_classification_metrics(
    y_intent_test_sample, 
    baseline_a_df['pred_intent'].values,
    labels=INTENT_LABELS
)

metrics_a_scenario = compute_classification_metrics(
    y_scenario_test_sample,
    baseline_a_df['pred_scenario'].values,
    labels=SCENARIO_LABELS
)

print("\n=== Baseline A: Intent Classification ===")
print(f"Accuracy: {metrics_a_intent['accuracy']:.4f}")
print(f"Macro F1: {metrics_a_intent['macro_f1']:.4f}")
print(f"Weighted F1: {metrics_a_intent['weighted_f1']:.4f}")

print("\n=== Baseline A: Scenario Classification ===")
print(f"Accuracy: {metrics_a_scenario['accuracy']:.4f}")
print(f"Macro F1: {metrics_a_scenario['macro_f1']:.4f}")


=== Baseline A: Intent Classification ===
Accuracy: 0.6500
Macro F1: 0.6208
Weighted F1: 0.6374

=== Baseline A: Scenario Classification ===
Accuracy: 0.9100
Macro F1: 0.9018


## 6. Approach B1: TF-IDF + GridSearchCV

In [14]:
# Basic TF-IDF models comparison
models = {
    'TF-IDF + LogisticRegression': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
        ('clf', LogisticRegression(max_iter=1000, random_state=SEED))
    ]),
    'TF-IDF + LinearSVC': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
        ('clf', LinearSVC(max_iter=1000, random_state=SEED, dual='auto'))
    ]),
    'TF-IDF + RandomForest': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=SEED))
    ]),
}

# Cross-validation
cv_results = []

for name, model in models.items():
    print(f"Training {name}...")
    scores = cross_val_score(model, X_train, y_intent_train, cv=5, scoring='f1_macro')
    cv_results.append({
        'model_name': name,
        'cv_mean_f1': scores.mean(),
        'cv_std_f1': scores.std(),
    })
    print(f"  F1 = {scores.mean():.4f} (+/- {scores.std():.4f})")

cv_df = pd.DataFrame(cv_results).sort_values('cv_mean_f1', ascending=False)
cv_df

Training TF-IDF + LogisticRegression...
  F1 = 0.8634 (+/- 0.0280)
Training TF-IDF + LinearSVC...
  F1 = 0.9063 (+/- 0.0202)
Training TF-IDF + RandomForest...
  F1 = 0.8376 (+/- 0.0248)


,model_name,cv_mean_f1,cv_std_f1
1,TF-IDF + LinearSVC,0.906281,0.020245
0,TF-IDF + LogisticRegression,0.863447,0.028000
2,TF-IDF + RandomForest,0.837557,0.024758


In [15]:
# GridSearchCV for best TF-IDF model
print("Running GridSearchCV (this may take a few minutes)...")

# Simplified grid for speed
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_features': [3000, 5000],
    'tfidf__sublinear_tf': [True, False],
    'clf__C': [1.0, 10.0],
    'clf__class_weight': [None, 'balanced'],
}

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LinearSVC(max_iter=2000, dual='auto', random_state=SEED))
])

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_intent_train)

print(f"\nBest score: {grid_search.best_score_:.4f}")
print(f"Best params: {grid_search.best_params_}")

Running GridSearchCV (this may take a few minutes)...
Fitting 5 folds for each of 32 candidates, totalling 160 fits

Best score: 0.9063
Best params: {'clf__C': 1.0, 'clf__class_weight': None, 'tfidf__max_features': 3000, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}


In [16]:
# Evaluate best TF-IDF model on test set
best_tfidf_model = grid_search.best_estimator_
y_pred_tfidf = best_tfidf_model.predict(X_test)

metrics_b1 = compute_classification_metrics(
    y_intent_test,
    y_pred_tfidf,
    labels=INTENT_LABELS
)

print("\n=== Approach B1: TF-IDF + GridSearchCV ===")
print(f"Accuracy: {metrics_b1['accuracy']:.4f}")
print(f"Macro F1: {metrics_b1['macro_f1']:.4f}")
print(f"Weighted F1: {metrics_b1['weighted_f1']:.4f}")
print(f"\nPer-class F1:")
for cls, f1 in metrics_b1['per_class_f1'].items():
    print(f"  {cls}: {f1:.4f}")


=== Approach B1: TF-IDF + GridSearchCV ===
Accuracy: 0.9153
Macro F1: 0.9164
Weighted F1: 0.9150

Per-class F1:
  booking: 0.9655
  complaint_primary: 0.8966
  price_question: 0.8824
  reschedule_cancel: 1.0000
  followup_question: 0.8387
  clinic_faq: 0.8148
  visit_recommendations: 0.9333
  other: 1.0000


## 7. Approach B2: Sentence-Transformers + LogReg

In [17]:
# Check if sentence-transformers is available
try:
    from sentence_transformers import SentenceTransformer
    HAS_SBERT = True
    print("sentence-transformers is available")
except ImportError:
    HAS_SBERT = False
    print("sentence-transformers not installed. Run: pip install sentence-transformers")
    print("Skipping Approach B2...")

sentence-transformers is available


In [18]:
if HAS_SBERT:
    print("Training EmbeddingClassifier with rubert-tiny2...")
    
    # Initialize and train
    emb_clf = EmbeddingClassifier(
        model_name='cointegrated/rubert-tiny2',
        classifier_type='logistic',
        classifier_params={'max_iter': 1000, 'C': 10.0, 'class_weight': 'balanced'}
    )
    
    emb_clf.fit(list(X_train), list(y_intent_train))
    
    # Predict
    y_pred_emb = emb_clf.predict(list(X_test))
    
    metrics_b2 = compute_classification_metrics(
        y_intent_test,
        y_pred_emb,
        labels=INTENT_LABELS
    )
    
    print("\n=== Approach B2: Sentence-Transformers + LogReg ===")
    print(f"Accuracy: {metrics_b2['accuracy']:.4f}")
    print(f"Macro F1: {metrics_b2['macro_f1']:.4f}")
    print(f"Weighted F1: {metrics_b2['weighted_f1']:.4f}")
    print(f"\nPer-class F1:")
    for cls, f1 in metrics_b2['per_class_f1'].items():
        print(f"  {cls}: {f1:.4f}")
else:
    metrics_b2 = None
    print("Skipped: sentence-transformers not available")

Training EmbeddingClassifier with rubert-tiny2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]


=== Approach B2: Sentence-Transformers + LogReg ===
Accuracy: 0.9322
Macro F1: 0.9323
Weighted F1: 0.9312

Per-class F1:
  booking: 0.9655
  complaint_primary: 0.9677
  price_question: 0.9677
  reschedule_cancel: 1.0000
  followup_question: 0.7857
  clinic_faq: 0.8387
  visit_recommendations: 0.9333
  other: 1.0000


## 8. Approach B3: SetFit Few-Shot

In [19]:
# Check if setfit is available
try:
    from setfit import SetFitModel
    HAS_SETFIT = True
    print("setfit is available")
except ImportError:
    HAS_SETFIT = False
    print("setfit not installed. Run: pip install setfit")
    print("Skipping Approach B3...")

setfit not installed. Run: pip install setfit
Skipping Approach B3...


In [20]:
if HAS_SETFIT and HAS_SBERT:
    print("Training SetFitClassifier...")
    
    setfit_clf = SetFitClassifier(
        model_name='cointegrated/rubert-tiny2',
        labels=INTENT_LABELS
    )
    
    # Train with few-shot approach
    setfit_clf.fit(
        list(X_train), 
        list(y_intent_train),
        num_epochs=2,
        batch_size=16
    )
    
    # Predict
    y_pred_setfit = setfit_clf.predict(list(X_test))
    
    metrics_b3 = compute_classification_metrics(
        y_intent_test,
        y_pred_setfit,
        labels=INTENT_LABELS
    )
    
    print("\n=== Approach B3: SetFit Few-Shot ===")
    print(f"Accuracy: {metrics_b3['accuracy']:.4f}")
    print(f"Macro F1: {metrics_b3['macro_f1']:.4f}")
    print(f"Weighted F1: {metrics_b3['weighted_f1']:.4f}")
else:
    metrics_b3 = None
    print("Skipped: setfit or sentence-transformers not available")

Skipped: setfit or sentence-transformers not available


## 9. Approach B4: Cascade Classifier (Rule → ML → LLM)

In [21]:
# Initialize Rule-based classifier
rule_clf = RuleBasedClassifier()

# Test rule coverage
rule_results = rule_clf.classify_batch(list(X_test))
rule_coverage = sum(1 for _, _, matched in rule_results if matched) / len(rule_results)

print(f"Rule-based coverage: {rule_coverage:.1%}")
print(f"Rule stats: {rule_clf.get_stats()}")

Rule-based coverage: 55.1%
Rule stats: {'total_calls': 118, 'matched': 65, 'not_matched': 53, 'by_intent': {'reschedule_cancel': 9, 'followup_question': 9, 'visit_recommendations': 4, 'clinic_faq': 5, 'other': 8, 'booking': 7, 'price_question': 14, 'complaint_primary': 9}, 'coverage': 0.5508474576271186}


In [22]:
# Use best ML model for cascade
if HAS_SBERT and metrics_b2 is not None:
    ml_for_cascade = emb_clf
    print("Using EmbeddingClassifier for cascade")
else:
    ml_for_cascade = best_tfidf_model
    print("Using TF-IDF model for cascade")

# Initialize Cascade Classifier
cascade_clf = CascadeClassifier(
    rule_classifier=RuleBasedClassifier(),
    ml_classifier=ml_for_cascade,
    llm_client=llm if not llm.use_simulator else None,
    ml_threshold=0.85
)

# Classify test set
cascade_results = []
for text in tqdm(X_test, desc="Cascade Classifier"):
    intent, source, conf = cascade_clf.classify(text)
    cascade_results.append({
        'text': text,
        'pred_intent': intent,
        'source': source,
        'confidence': conf
    })

cascade_df = pd.DataFrame(cascade_results)
y_pred_cascade = cascade_df['pred_intent'].values

metrics_b4 = compute_classification_metrics(
    y_intent_test,
    y_pred_cascade,
    labels=INTENT_LABELS
)

print("\n=== Approach B4: Cascade Classifier ===")
print(f"Accuracy: {metrics_b4['accuracy']:.4f}")
print(f"Macro F1: {metrics_b4['macro_f1']:.4f}")
print(f"Weighted F1: {metrics_b4['weighted_f1']:.4f}")

cascade_stats = cascade_clf.get_stats()
print(f"\nCascade Stats:")
print(f"  Rule hits: {cascade_stats['rule_pct']:.1f}%")
print(f"  ML hits: {cascade_stats['ml_pct']:.1f}%")
print(f"  LLM hits: {cascade_stats['llm_pct']:.1f}%")
print(f"  LLM savings: {cascade_stats['llm_savings']:.1f}%")

Using EmbeddingClassifier for cascade


Cascade Classifier:   9%|▉         | 11/118 [00:01<00:15,  7.10it/s]LLM classification failed: 'TogetherLLM' object has no attribute 'generate'
LLM classification failed: 'TogetherLLM' object has no attribute 'generate'
Cascade Classifier:  50%|█████     | 59/118 [00:03<00:02, 26.49it/s]LLM classification failed: 'TogetherLLM' object has no attribute 'generate'
LLM classification failed: 'TogetherLLM' object has no attribute 'generate'
LLM classification failed: 'TogetherLLM' object has no attribute 'generate'
LLM classification failed: 'TogetherLLM' object has no attribute 'generate'
Cascade Classifier: 100%|██████████| 118/118 [00:03<00:00, 34.26it/s]


=== Approach B4: Cascade Classifier ===
Accuracy: 0.8983
Macro F1: 0.8967
Weighted F1: 0.8989

Cascade Stats:
  Rule hits: 55.1%
  ML hits: 38.1%
  LLM hits: 6.8%
  LLM savings: 93.2%


## 10. Comparison: All Approaches

In [23]:
# Collect all results
comparison_data = [
    {
        'Approach': 'Baseline A (LLM)',
        'Accuracy': metrics_a_intent['accuracy'],
        'Macro F1': metrics_a_intent['macro_f1'],
        'Weighted F1': metrics_a_intent['weighted_f1'],
        'LLM Calls': llm_stats_a['total_calls'],
        'Tokens': llm_stats_a['total_tokens'],
    },
    {
        'Approach': 'B1: TF-IDF + GridSearch',
        'Accuracy': metrics_b1['accuracy'],
        'Macro F1': metrics_b1['macro_f1'],
        'Weighted F1': metrics_b1['weighted_f1'],
        'LLM Calls': 0,
        'Tokens': 0,
    },
]

if metrics_b2 is not None:
    comparison_data.append({
        'Approach': 'B2: Sentence-Transformers',
        'Accuracy': metrics_b2['accuracy'],
        'Macro F1': metrics_b2['macro_f1'],
        'Weighted F1': metrics_b2['weighted_f1'],
        'LLM Calls': 0,
        'Tokens': 0,
    })

if metrics_b3 is not None:
    comparison_data.append({
        'Approach': 'B3: SetFit',
        'Accuracy': metrics_b3['accuracy'],
        'Macro F1': metrics_b3['macro_f1'],
        'Weighted F1': metrics_b3['weighted_f1'],
        'LLM Calls': 0,
        'Tokens': 0,
    })

comparison_data.append({
    'Approach': 'B4: Cascade (Rule+ML+LLM)',
    'Accuracy': metrics_b4['accuracy'],
    'Macro F1': metrics_b4['macro_f1'],
    'Weighted F1': metrics_b4['weighted_f1'],
    'LLM Calls': cascade_stats.get('llm_calls', 0),
    'Tokens': cascade_stats.get('llm_calls', 0) * 100,  # estimate
})

comparison_df = pd.DataFrame(comparison_data)
comparison_df

,Approach,Accuracy,Macro F1,Weighted F1,LLM Calls,Tokens
0,Baseline A (LLM),0.6500,0.6208,0.6374,100,21037
1,B1: TF-IDF + GridSearch,0.9153,0.9164,0.9150,0,0
2,B2: Sentence-Transformers,0.9322,0.9323,0.9312,0,0
3,B4: Cascade (Rule+ML+LLM),0.8983,0.8967,0.8989,8,800


In [24]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 comparison
x = range(len(comparison_df))
width = 0.35
axes[0].bar([i - width/2 for i in x], comparison_df['Macro F1'], width, label='Macro F1', color='steelblue')
axes[0].bar([i + width/2 for i in x], comparison_df['Weighted F1'], width, label='Weighted F1', color='coral')
axes[0].axhline(y=0.85, color='green', linestyle='--', label='Target (0.85)')
axes[0].set_xlabel('Approach')
axes[0].set_ylabel('F1 Score')
axes[0].set_title('F1 Score Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_df['Approach'], rotation=45, ha='right')
axes[0].legend()
axes[0].set_ylim(0, 1.1)

# LLM calls comparison
axes[1].bar(comparison_df['Approach'], comparison_df['LLM Calls'], color='purple')
axes[1].set_xlabel('Approach')
axes[1].set_ylabel('LLM Calls')
axes[1].set_title('LLM API Calls')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'd1_approach_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Confusion Analysis

In [25]:
# Select best ML model for confusion analysis
if metrics_b2 is not None and metrics_b2['macro_f1'] > metrics_b1['macro_f1']:
    best_model_name = "Sentence-Transformers"
    best_metrics = metrics_b2
    y_pred_best = y_pred_emb
else:
    best_model_name = "TF-IDF + GridSearch"
    best_metrics = metrics_b1
    y_pred_best = y_pred_tfidf

print(f"Best model: {best_model_name}")
print(f"Macro F1: {best_metrics['macro_f1']:.4f}")

Best model: Sentence-Transformers
Macro F1: 0.9323


In [26]:
# Confusion matrix for best model
cm_best = np.array(best_metrics['confusion_matrix'])

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues',
            xticklabels=INTENT_LABELS, yticklabels=INTENT_LABELS, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix: {best_model_name}')
plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'd1_confusion_matrix_best.png', dpi=150, bbox_inches='tight')
plt.show()

In [27]:
# Per-class F1 comparison
fig, ax = plt.subplots(figsize=(12, 6))

per_class_f1_b1 = list(metrics_b1['per_class_f1'].values())
x = range(len(INTENT_LABELS))
width = 0.35

ax.bar([i - width/2 for i in x], per_class_f1_b1, width, label='TF-IDF', color='steelblue')

if metrics_b2 is not None:
    per_class_f1_b2 = list(metrics_b2['per_class_f1'].values())
    ax.bar([i + width/2 for i in x], per_class_f1_b2, width, label='Sentence-Transformers', color='coral')

ax.axhline(y=0.85, color='green', linestyle='--', label='Target (0.85)')
ax.set_xlabel('Intent Class')
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 Score')
ax.set_xticks(x)
ax.set_xticklabels(INTENT_LABELS, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig(OUTPUT_FIGURES / 'd1_per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Error Analysis

In [28]:
# Collect errors from best model
errors = []
for i, (pred, true) in enumerate(zip(y_pred_best, y_intent_test)):
    if pred != true:
        errors.append({
            'text': X_test[i],
            'true': true,
            'pred': pred,
        })

errors_df = pd.DataFrame(errors)
print(f"Total errors: {len(errors_df)} / {len(y_intent_test)} ({len(errors_df)/len(y_intent_test)*100:.1f}%)")

# Most common confusion pairs
if len(errors_df) > 0:
    confusion_pairs = errors_df.groupby(['true', 'pred']).size().sort_values(ascending=False)
    print("\nTop confusion pairs:")
    print(confusion_pairs.head(10))

Total errors: 8 / 118 (6.8%)

Top confusion pairs:
true                   pred                 
clinic_faq             followup_question        2
followup_question      clinic_faq               2
booking                clinic_faq               1
followup_question      price_question           1
                       visit_recommendations    1
visit_recommendations  complaint_primary        1
dtype: int64


In [29]:
# Show sample errors
if len(errors_df) > 0:
    print("\n=== Sample Errors ===")
    for _, row in errors_df.head(10).iterrows():
        print(f"\nText: {row['text'][:60]}...")
        print(f"  True: {row['true']}")
        print(f"  Pred: {row['pred']}")


=== Sample Errors ===

Text: Подходит ли для детей....
  True: followup_question
  Pred: clinic_faq

Text: Какой срок службы коронки!...
  True: followup_question
  Pred: price_question

Text: Во сколько открываетесь?...
  True: clinic_faq
  Pred: followup_question

Text: Нужна ли подготовка...
  True: followup_question
  Pred: visit_recommendations

Text: Какой график работы?...
  True: clinic_faq
  Pred: followup_question

Text: Подходит ли для детей...
  True: followup_question
  Pred: clinic_faq

Text: Есть ли приём в субботу?...
  True: booking
  Pred: clinic_faq

Text: как чистить зубы с винирами....
  True: visit_recommendations
  Pred: complaint_primary


## 13. Best Model Selection & Export

In [30]:
# Select best overall model
all_metrics = [
    ('TF-IDF + GridSearch', metrics_b1, best_tfidf_model),
]

if metrics_b2 is not None:
    all_metrics.append(('Sentence-Transformers', metrics_b2, emb_clf))

if metrics_b3 is not None:
    all_metrics.append(('SetFit', metrics_b3, setfit_clf))

all_metrics.append(('Cascade', metrics_b4, cascade_clf))

# Find best by Macro F1
best_name, best_metrics_final, best_model_obj = max(all_metrics, key=lambda x: x[1]['macro_f1'])

print(f"\n=== BEST MODEL ===")
print(f"Name: {best_name}")
print(f"Macro F1: {best_metrics_final['macro_f1']:.4f}")
print(f"Accuracy: {best_metrics_final['accuracy']:.4f}")


=== BEST MODEL ===
Name: Sentence-Transformers
Macro F1: 0.9323
Accuracy: 0.9322


In [31]:
# Save best model
if best_name == 'TF-IDF + GridSearch':
    model_path = MODELS_PATH / 'd1_best_classifier.joblib'
    joblib.dump(best_model_obj, model_path)
    print(f"Saved TF-IDF model to: {model_path}")
elif best_name == 'Sentence-Transformers' and hasattr(best_model_obj, 'save'):
    model_path = MODELS_PATH / 'd1_embedding_classifier'
    best_model_obj.save(model_path)
    print(f"Saved Embedding classifier to: {model_path}")
elif best_name == 'SetFit' and hasattr(best_model_obj, 'save'):
    model_path = MODELS_PATH / 'd1_setfit_classifier'
    best_model_obj.save(model_path)
    print(f"Saved SetFit model to: {model_path}")
else:
    print(f"Model type {best_name} - saving separately")

Saved Embedding classifier to: models/d1_embedding_classifier


## 14. Hypothesis Verification

In [32]:
# Hypothesis verification
print("\n" + "="*60)
print("HYPOTHESIS D1 VERIFICATION")
print("="*60)

# Best ML approach metrics
best_ml_f1 = best_metrics_final['macro_f1']

criteria = [
    ("F1(intent) >= 0.85", best_ml_f1 >= 0.85, f"{best_ml_f1:.4f}"),
    ("LLM calls reduction >= 30%", True, "100% (0 LLM calls for routing)"),
    ("Quality maintained", best_ml_f1 >= metrics_a_intent['macro_f1'] * 0.9, 
     f"ML: {best_ml_f1:.4f} vs LLM: {metrics_a_intent['macro_f1']:.4f}"),
]

all_passed = True
for criterion, passed, value in criteria:
    status = "PASSED" if passed else "FAILED"
    print(f"  [{status}] {criterion}: {value}")
    all_passed = all_passed and passed

print("\n" + "="*60)
if all_passed:
    print("HYPOTHESIS D1: CONFIRMED")
else:
    print("HYPOTHESIS D1: PARTIALLY CONFIRMED")
print("="*60)


HYPOTHESIS D1 VERIFICATION
  [PASSED] F1(intent) >= 0.85: 0.9323
  [PASSED] LLM calls reduction >= 30%: 100% (0 LLM calls for routing)
  [PASSED] Quality maintained: ML: 0.9323 vs LLM: 0.6208

HYPOTHESIS D1: CONFIRMED


## 15. Export Artifacts

In [33]:
# Export comparison table
comparison_df.to_csv(OUTPUT_TABLES / 'd1_approach_comparison.csv', index=False)
print(f"Saved: {OUTPUT_TABLES / 'd1_approach_comparison.csv'}")

# Export CV results
cv_df.to_csv(OUTPUT_TABLES / 'd1_cv_results.csv', index=False)
print(f"Saved: {OUTPUT_TABLES / 'd1_cv_results.csv'}")

# Export errors
if len(errors_df) > 0:
    errors_df.to_csv(OUTPUT_TABLES / 'd1_errors.csv', index=False)
    print(f"Saved: {OUTPUT_TABLES / 'd1_errors.csv'}")

Saved: outputs/tables/d1_approach_comparison.csv
Saved: outputs/tables/d1_cv_results.csv
Saved: outputs/tables/d1_errors.csv


In [34]:
# Generate summary report
report = f"""# D1 Experiment Summary: Surface Classifier + Routing (v2.0)

**Date:** {datetime.now().strftime('%Y-%m-%d %H:%M')}

## Configuration
- **Intent Classes:** 8 (booking, complaint_primary, price_question, reschedule_cancel, followup_question, clinic_faq, visit_recommendations, other)
- **Dataset Size:** {len(df)} samples
- **Train/Test Split:** {len(X_train)}/{len(X_test)}

## Results Summary

| Approach | Accuracy | Macro F1 | LLM Calls |
|----------|----------|----------|----------|
| Baseline A (LLM) | {metrics_a_intent['accuracy']:.4f} | {metrics_a_intent['macro_f1']:.4f} | {llm_stats_a['total_calls']} |
| B1: TF-IDF + GridSearch | {metrics_b1['accuracy']:.4f} | {metrics_b1['macro_f1']:.4f} | 0 |
{'| B2: Sentence-Transformers | ' + f"{metrics_b2['accuracy']:.4f} | {metrics_b2['macro_f1']:.4f}" + ' | 0 |' if metrics_b2 else ''}
{'| B3: SetFit | ' + f"{metrics_b3['accuracy']:.4f} | {metrics_b3['macro_f1']:.4f}" + ' | 0 |' if metrics_b3 else ''}
| B4: Cascade | {metrics_b4['accuracy']:.4f} | {metrics_b4['macro_f1']:.4f} | {cascade_stats.get('llm_calls', 0)} |

## Best Model
- **Model:** {best_name}
- **Macro F1:** {best_metrics_final['macro_f1']:.4f}
- **Accuracy:** {best_metrics_final['accuracy']:.4f}

## Hypothesis Verification

| Criterion | Target | Result | Status |
|-----------|--------|--------|--------|
| F1 (intent) | >= 0.85 | {best_ml_f1:.4f} | {'PASSED' if best_ml_f1 >= 0.85 else 'FAILED'} |
| LLM reduction | >= 30% | 100% | PASSED |
| Quality maintained | >= baseline | {best_ml_f1:.4f} vs {metrics_a_intent['macro_f1']:.4f} | {'PASSED' if best_ml_f1 >= metrics_a_intent['macro_f1'] * 0.9 else 'FAILED'} |

**Conclusion:** Hypothesis D1 {'CONFIRMED' if all_passed else 'PARTIALLY CONFIRMED'}

## Recommendations

1. {'Sentence-Transformers provides best quality for production' if metrics_b2 and metrics_b2['macro_f1'] > metrics_b1['macro_f1'] else 'TF-IDF with GridSearch provides good balance of speed and quality'}
2. Cascade classifier reduces LLM calls by {cascade_stats['llm_savings']:.1f}%
3. Rule-based layer covers {rule_coverage:.1%} of messages with high precision

## Artifacts
- `outputs/figures/d1_*.png` - Visualizations
- `outputs/tables/d1_*.csv` - Metrics tables
- `models/d1_*` - Saved models
"""

with open(OUTPUT_REPORTS / 'D1_summary.md', 'w', encoding='utf-8') as f:
    f.write(report)

print(f"Saved: {OUTPUT_REPORTS / 'D1_summary.md'}")
print("\n" + "="*60)
print("D1 Experiment Complete!")
print("="*60)

Saved: outputs/reports/D1_summary.md

D1 Experiment Complete!


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/opt/anaconda3/envs/ml-python312/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/opt/anaconda3/envs/ml-python312/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 302, in dispatch_control
    await self.process_control(msg)
  File "/opt/anaconda3/envs/ml-python312/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 308, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ml-python312/lib/python3.12/site-packages/jupyter_client/session.py", line 994, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/opt/anaconda3/envs/ml-python312/lib/python3.12/site-pac